In [2]:
%pip install pyspark==3.5.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 2.2 MB/s eta 0:00:0000:0100:03
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 2.5 MB/s eta 0:00:00a 0:00:01
  Created wheel for pyspark: filename=pyspark-3.5.3-py2.py3-none-any.whl size=317840630 sha256=a381dd61c5576b092ddf9723b7607c765a68f674ba8632cb1e1f714303c8e8f5
  Stored in directory: /tmp/pip-ephem-wheel-cache-ozvv02p9/wheels/97/f5/c0/947e2c0942b361ffe58651f36bd7f13772675b3863fd63d1b1
Successfully built pyspark
Note: you may need to restart the kernel to use updated packages.


### Укажем компилятор java для работы spark'a.

In [1]:
# Дирекстория, в которой pyspark ище jav'у.
!echo $JAVA_HOME
# Просмотрели все имеющиеся файлы в нем.
!ls /usr/lib/jvm
# Убедились в версии java'ы
!java -version

/usr/lib/jvm/java-8-openjdk-amd64
java-1.17.0-openjdk-amd64  java-1.8.0-openjdk-amd64
java-17-openjdk-amd64	   java-8-openjdk-amd64
openjdk version "17.0.8.1" 2023-08-24
OpenJDK Runtime Environment (build 17.0.8.1+1-Ubuntu-0ubuntu122.04)
OpenJDK 64-Bit Server VM (build 17.0.8.1+1-Ubuntu-0ubuntu122.04, mixed mode, sharing)


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

In [3]:
!hdfs dfs -put -f top5_local.txt hdfs://namenode:8020/hw2/top_5/mr_result

put: `hdfs://namenode:8020/hw2/top_5/mr_result': No such file or directory


In [4]:
import pyspark

In [5]:
from pyspark.sql import SparkSession
# Создали spark сессию и дали ей имя 
spark = SparkSession.builder.appName('home_work_2').getOrCreate() 

In [6]:
# Можно увидеть, что мастер запустился локально, имя прокинулось.

In [7]:
spark

Считаем данные из файла.

In [10]:
!which hdfs

/opt/hadoop/bin/hdfs


In [11]:
!hdfs dfs -mkdir -p hdfs://namenode:8020/hw2/wordcount_hw/input
!hdfs dfs -put -f bible.txt hdfs://namenode:8020/hw2/wordcount_hw/input

In [12]:
sc = spark.sparkContext
rdd_text = sc.textFile("hdfs://namenode:8020/hw2/wordcount_hw/input/bible.txt")
rdd_text.take(5)

['In the beginning God created the heaven and the earth. And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters. ',
 'And God said, Let there be light: and there was light. ',
 'And God saw the light, that it was good: and God divided the light from the darkness. ',
 'And God called the light Day, and the darkness he called Night. And the evening and the morning were the first day. ',
 'And God said, Let there be a firmament in the midst of the waters, and let it divide the waters from the waters. ']

In [15]:
# Разобьем текст по предложениям на слова.
# rdd_text.flatMap(lambda s: s.split()) 
# flatMap дробит весь текст на отдельные слова.
# Подсчитаем кол-во слов при помощи flatMap + map + reduceByKey

import re
word_count = (
    rdd_text
    .flatMap(lambda s: s.split())
    .filter(lambda w: len(w) > 4)
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b)
)

sorted_word_count = word_count.sortBy(lambda i: i[0])
sorted_word_count.take(10)

[('(According', 1),
 ('(After', 1),
 ('(Also', 1),
 ('(Beforetime', 1),
 ('(Howbeit', 1),
 ('(That', 1),
 ('(Then', 1),
 ('(There', 1),
 ('(Though', 1),
 ('(Touch', 1)]

Поместим файлс ответом из первой домашки для word_count и проверим их соответствие. 

In [16]:
!hdfs dfs -put -f wordcount1_local.txt hdfs://namenode:8020/hw2/wordcount_hw

In [17]:
mr_result = (
    sc.textFile("hdfs://namenode:8020/hw2/wordcount_hw/wordcount1_local.txt")
      .map(lambda line: line.rsplit("\t", 1))
      .map(lambda p: (p[0], int(p[1])))
)

mr_result.take(10)

[('Aaron', 205),
 ('Aaron,', 95),
 ('Aaron.', 11),
 ('Aaron:', 2),
 ('Aaron;', 5),
 ('Aaron?', 1),
 ('Aaronites,', 2),
 ("Aaron's", 31),
 ('Abaddon,', 1),
 ('Abagtha,', 1)]

In [18]:
all_pairs = word_count.count()


In [19]:
mr_pairs = mr_result.count()


In [21]:
# Сохраняем результаты в памяти, чтобы не пересчитывать их для каждого действия
word_count.cache()
mr_result.cache()
all_pairs = word_count.count()
mr_pairs = mr_result.count()
# join соединяет два RDD по ключу (слову): для каждого слова, которое есть в обоих,
# получается пара (слово, (кол-во в RDD, кол-во в MapReduce)).
# filter оставляет только слова, у которых количества совпадают.
matches = (
           word_count.join(mr_result)
              .filter(lambda kv: kv[1][0] == kv[1][1])
              .count()
           )
print(f"Кол-во пар, полученных с помощью RDD: {all_pairs}")
print(f"Кол-во пар, полученных с помощью MapReduce: {mr_pairs}")
print(f"Кол-во совпадающих пар: {matches}")

Кол-во пар, полученных с помощью RDD: 26280
Кол-во пар, полученных с помощью MapReduce: 26280
Кол-во совпадающих пар: 26280


Решим задачу вторую при помощи RDD.

Поместим файл в hdfs.

In [22]:
!hdfs dfs -mkdir -p hdfs://namenode:8020/hw2/top_5/input
!hdfs dfs -put -f Poseschenia_sai_774_tov.csv hdfs://namenode:8020/hw2/top_5/input

In [23]:
rdd_visits = sc.textFile("hdfs://namenode:8020/hw2/top_5/input/Poseschenia_sai_774_tov.csv")
rdd_visits.take(5)

['https://gonzales-bautista.com/;2024-05-28 05:58:33.853479',
 'https://gonzales-bautista.com/;2024-05-26 03:44:36.853479',
 'https://www.davis-berg.com/;2024-05-31 17:22:01.853479',
 'https://www.montoya.com/;2024-05-29 13:41:35.853479',
 'https://gonzales-bautista.com/;2024-06-01 06:49:16.853479']

Разделим данные на 3 поля: url, дата, мусор. Мусор выкинем, а в качестве ключей используем дату и по ней сортируем.

In [24]:
rdd_mapperd = rdd_visits.map(lambda s: s.split()[0]).map(
                                                        lambda url_data:(
                                                                        (url_data.split(';')[1], 
                                                                        url_data.split(';')[0]),
                                                                        1
                                                                        )
                                                        ).sortBy(lambda i: i[0])
rdd_mapperd.take(5)

[(('2024-05-26', 'http://abbott-brown.com/'), 1),
 (('2024-05-26', 'http://abbott-ellis.com/'), 1),
 (('2024-05-26', 'http://abbott-hall.com/'), 1),
 (('2024-05-26', 'http://abbott-holden.com/'), 1),
 (('2024-05-26', 'http://abbott-weaver.biz/'), 1)]

In [25]:
rdd_reduced = rdd_mapperd.reduceByKey(lambda a, b: a + b)

# Делаем ключом дату: (дата, (количество, сайт))
by_date = rdd_reduced.map(lambda x: (x[0][0], (x[1], x[0][1])))

def top5(items):
    return sorted(items, key=lambda x: (-x[0], x[1]))[:5]

top5_by_date = by_date.aggregateByKey([], lambda a, b: top5(a + [b]), lambda a, b: top5(a + b))
top5_by_date.take(5)

[('2024-05-30',
  [(353, 'https://gonzales-bautista.com/'),
   (246, 'https://smith.com/'),
   (239, 'https://www.smith.com/'),
   (229, 'http://smith.com/'),
   (225, 'http://www.smith.com/')]),
 ('2024-05-31',
  [(374, 'https://gonzales-bautista.com/'),
   (244, 'https://www.smith.com/'),
   (228, 'http://smith.com/'),
   (221, 'http://www.smith.com/'),
   (219, 'https://smith.com/')]),
 ('2024-06-02',
  [(7, 'http://smith.com/'),
   (7, 'https://gonzales-bautista.com/'),
   (6, 'https://www.williams.com/'),
   (5, 'http://lee.com/'),
   (5, 'http://miller.com/')]),
 ('2024-05-26',
  [(335, 'https://gonzales-bautista.com/'),
   (235, 'http://smith.com/'),
   (221, 'https://www.smith.com/'),
   (212, 'http://www.smith.com/'),
   (212, 'https://smith.com/')]),
 ('2024-05-27',
  [(376, 'https://gonzales-bautista.com/'),
   (270, 'https://www.smith.com/'),
   (236, 'https://smith.com/'),
   (215, 'http://smith.com/'),
   (208, 'http://www.smith.com/')])]

In [26]:
!hdfs dfs -mkdir -p hdfs://namenode:8020/hw2/top_5/mr_result
!hdfs dfs -put -f top5_local.txt hdfs://namenode:8020/hw2/top_5/mr_result

Чтобы сравнить результаты работы spark и mapreduce приведем их к одному виду.

In [27]:
spark_top = top5_by_date.flatMap(lambda kv: [(kv[0], site, count) for count, site in kv[1]])


mr_top = (
            sc.textFile("hdfs://namenode:8020/hw2/top_5/mr_result")
            .map(lambda line: line.split("\t"))
            .map(lambda p: (p[0], p[1], int(p[2])))
         )

In [28]:
# Сохранили результат в кеше и свели все к одной партиции, так как кол-во строк небольшое (40)
spark_top = spark_top.coalesce(1).cache()
mr_top = mr_top.coalesce(1).cache()
print("Кол-во строк в результате работы Spark:", spark_top.count())
print("Кол-во строк в результате работы MapReduce:", mr_top.count())
only_in_spark = spark_top.subtract(mr_top, numPartitions=1)
only_in_mr = mr_top.subtract(spark_top, numPartitions=1)
print("Строки, имеющиеся только в Spark:", only_in_spark.collect())
print("Строки, имеющиеся только в MapReduce:", only_in_mr.collect())

Кол-во строк в результате работы Spark: 40
Кол-во строк в результате работы MapReduce: 40
Строки, имеющиеся только в Spark: []
Строки, имеющиеся только в MapReduce: []


### DataFramAPI

Задача 1 с word count.

In [29]:
df_text = spark.read.text("hdfs://namenode:8020/hw2/wordcount_hw/input/bible.txt")

In [30]:
# Посмотрим что из себя представляют данные.
df_text.printSchema()
df_text.show(5, truncate=50)

root
 |-- value: string (nullable = true)

+--------------------------------------------------+
|                                             value|
+--------------------------------------------------+
|In the beginning God created the heaven and the...|
|And God said, Let there be light: and there was...|
|And God saw the light, that it was good: and Go...|
|And God called the light Day, and the darkness ...|
|And God said, Let there be a firmament in the m...|
+--------------------------------------------------+
only showing top 5 rows



In [31]:
from pyspark.sql import functions as F

In [35]:
# Разобьем каждую строку на слова.
# F.col("value") берет столбец value.
# .alias("words") задали имя новой колонке "words".
# df_text.select выбрали колонку для новой таблицы.
df_new = df_text.select(
    F.split(F.col("value"), r'\s+').alias("words")
)
# F.explode() разделили каждую строку на отдельные слова и из них составили новые строки
df_words = df_new.select(F.explode(F.col("words")).alias("word"))
df_words.show(10)

+---------+
|     word|
+---------+
|       In|
|      the|
|beginning|
|      God|
|  created|
|      the|
|   heaven|
|      and|
|      the|
|   earth.|
+---------+
only showing top 10 rows



In [36]:
# Применили фильтр,чтобы убрать слова длиною меньше 4.
# F.length() возвращает длину каждой строки.
df_filtered = df_words.filter(F.length(F.col("word")) > 4)
# df_filtered.groupBy("word") собрали все строки с одинаковыми элементами.
# .count() считает кол-во строк в группе
df_count = df_filtered.groupBy("word").count()
# df_count.orderBy("word") отсортировали по колонке "word"
df_result = df_count.orderBy("word")
df_result.show(10)

+-----------+-----+
|       word|count|
+-----------+-----+
| (According|    1|
|     (After|    1|
|      (Also|    1|
|(Beforetime|    1|
|   (Howbeit|    1|
|      (That|    1|
|      (Then|    1|
|     (There|    1|
|    (Though|    1|
|     (Touch|    1|
+-----------+-----+
only showing top 10 rows



Сверим полученный результат с MapReduce и RDD

In [39]:
# Считали результат полученный от mapreduce и преобразовали его в dataframe.
df_mr = (
    spark.read.text("hdfs://namenode:8020/hw2/wordcount_hw/wordcount1_local.txt")
    .select(F.split(F.col("value"), "\t").alias("parts"))
    .select(
           F.col("parts")[0].alias("word"),
           F.col("parts")[1].cast("long").alias("count"),
           )
        )

df_mr.show(5)

+------+-----+
|  word|count|
+------+-----+
| Aaron|  205|
|Aaron,|   95|
|Aaron.|   11|
|Aaron:|    2|
|Aaron;|    5|
+------+-----+
only showing top 5 rows



In [43]:
# Конвертировали rdd в dataframe для сравнения.
df_rdd = word_count.toDF(["word", "count"])

# Сохраняем все три результата в памяти, чтобы не пересчитывать их для каждого сравнения.
df_result.cache()
df_mr.cache()
df_rdd.cache()

print("Слов в dataframe:", df_result.count())
print("Слов в mapreduce:", df_mr.count())
print("Слов в rdd:", df_rdd.count())

# exceptAll возвращает строки первой таблицы, которых нет во второй
print("Только в dataframe:", df_result.exceptAll(df_mr).count())
print("Только в mapreduce:", df_mr.exceptAll(df_result).count())
print("Только в dataframe:", df_result.exceptAll(df_rdd).count())
print("Только в rdd:", df_rdd.exceptAll(df_result).count())

Слов в dataframe: 26280
Слов в mapreduce: 26280
Слов в rdd: 26280
Только в dataframe: 0
Только в mapreduce: 0
Только в dataframe: 0
Только в rdd: 0


### Топ 5 посещаемых сайтов при помощи DataFrame

In [44]:
df_visits = spark.read.csv(
    "hdfs://namenode:8020/hw2/top_5/input/Poseschenia_sai_774_tov.csv",
    sep=";",
    schema="url STRING, visit_time STRING",
)

df_visits.printSchema()
df_visits.show(5, truncate=False)

root
 |-- url: string (nullable = true)
 |-- visit_time: string (nullable = true)

+------------------------------+--------------------------+
|url                           |visit_time                |
+------------------------------+--------------------------+
|https://gonzales-bautista.com/|2024-05-28 05:58:33.853479|
|https://gonzales-bautista.com/|2024-05-26 03:44:36.853479|
|https://www.davis-berg.com/   |2024-05-31 17:22:01.853479|
|https://www.montoya.com/      |2024-05-29 13:41:35.853479|
|https://gonzales-bautista.com/|2024-06-01 06:49:16.853479|
+------------------------------+--------------------------+
only showing top 5 rows



Приведем к удобному формату dataframe. Оставим только дату, без точного времени.

In [49]:
# F.trim() - убирает лишние пробелы.
df_new = df_visits.select(
                         F.split(F.trim(F.col('visit_time')), ' ')[0].alias("date"),
                         F.trim(F.col("url")).alias("url")
                         )
df_new.show(5, truncate=False)

+----------+------------------------------+
|date      |url                           |
+----------+------------------------------+
|2024-05-28|https://gonzales-bautista.com/|
|2024-05-26|https://gonzales-bautista.com/|
|2024-05-31|https://www.davis-berg.com/   |
|2024-05-29|https://www.montoya.com/      |
|2024-06-01|https://gonzales-bautista.com/|
+----------+------------------------------+
only showing top 5 rows



Посчитаем кол-во одинаковых элементов по ключу (дата, урл)

In [51]:
# Сгруппируем при помощи .groupBy элементы с одинаковыми урл и датой и посчитаем их кол-во. 
df_site_counts = df_new.groupBy("date", "url").count()
df_site_counts.show(5, truncate=False)

+----------+-------------------------+-----+
|date      |url                      |count|
+----------+-------------------------+-----+
|2024-06-01|https://reed-padilla.com/|12   |
|2024-05-29|http://guzman.org/       |6    |
|2024-05-28|https://www.russo.org/   |1    |
|2024-06-01|http://fleming.com/      |15   |
|2024-05-28|https://mack.com/        |12   |
+----------+-------------------------+-----+
only showing top 5 rows



In [63]:
from pyspark.sql import Window

# Window.partitionBy("date") задали окно, для того чтобы разделить элементы на окна по дате.
# F.col("count").desc() сортируем сперва по кол-ву на убывание
# F.col("url").asc() если кол-во одинаково, то сортируем по урл в поорядке возростанияю
w = Window.partitionBy("date").orderBy(F.col("count").desc(), F.col("url").asc())
# withColumn("rank", F.row_number().over(w)) создали колонку rank и нумеруем строки по правилу окна w.
# .filter(F.col("rank") <= 5) оставили 5 самых просматриваемых сайтов. 
df_top5 = (
            df_site_counts
            .withColumn("rank", F.row_number().over(w))
            .filter(F.col("rank") <= 5) 
            .drop("rank") # удалили колонку rank
            .orderBy("date", F.col("count").desc(), "url") # отсортировали таблицу по колонкам date,count,url.
          )

df_top5.show(40, truncate=False)

+----------+------------------------------+-----+
|date      |url                           |count|
+----------+------------------------------+-----+
|2024-05-26|https://gonzales-bautista.com/|335  |
|2024-05-26|http://smith.com/             |235  |
|2024-05-26|https://www.smith.com/        |221  |
|2024-05-26|http://www.smith.com/         |212  |
|2024-05-26|https://smith.com/            |212  |
|2024-05-27|https://gonzales-bautista.com/|376  |
|2024-05-27|https://www.smith.com/        |270  |
|2024-05-27|https://smith.com/            |236  |
|2024-05-27|http://smith.com/             |215  |
|2024-05-27|http://www.smith.com/         |208  |
|2024-05-28|https://gonzales-bautista.com/|368  |
|2024-05-28|https://smith.com/            |256  |
|2024-05-28|https://www.smith.com/        |251  |
|2024-05-28|http://smith.com/             |224  |
|2024-05-28|http://www.smith.com/         |204  |
|2024-05-29|https://gonzales-bautista.com/|402  |
|2024-05-29|https://www.smith.com/        |242  |


Проверим полученный результат.

In [66]:
rows_df = set(tuple(row) for row in df_top5.collect())
rows_mr = set(
    (p[0], p[1], int(p[2]))
    for p in (
             line.split("\t") for line in sc.textFile(
                                                       "hdfs://namenode:8020/hw2/top_5/mr_result"
                                                       ).collect()
             )
             )

rows_rdd = set(spark_top.collect())
# Посчитаем еол-во элементов в каждом из решений и определим элементы, которые различаются.
print("Строк в dataframe:", len(rows_df))
print("Строк в mapreduce:", len(rows_mr))
print("Строк в rdd:", len(rows_rdd))
print("Только в dataframe:", rows_df - rows_mr)
print("Только в mapreduce:", rows_mr - rows_df)
print("Только в dataframe:", rows_df - rows_rdd)
print("Только в rdd:", rows_rdd - rows_df)

Строк в dataframe: 40
Строк в mapreduce: 40
Строк в rdd: 40
Только в dataframe: set()
Только в mapreduce: set()
Только в dataframe: set()
Только в rdd: set()


Проверка показала, что ответ совпадает с проршлыми результатами.